### Import Dependencies

In [1]:
import openai
import pandas as pd
import tiktoken

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import FieldCondition, MatchAny, Filter
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

### Create Qdrant Collection for hybrid Search

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [3]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    }
)

True

In [4]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [5]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [6]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon Reviews Data

In [9]:
df_reviews = pd.read_json("../../data/Electornics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [10]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True
1,5,Great so far...,"Speedy delivery, great sound and a great warra...",[],B08H1WNYTR,B0C72D4J46,AFANVB6MPHJTCTFOVIEBKLWZ2GVA,2022-07-05 14:40:48.001,0,True
2,5,Set up is not that easy.,Nice looking set but installation instructions...,[],B09XGX1GMJ,B09XGQQ98K,AHACLF2COQQE2V33ZFXQ7THZOJ2Q,2022-09-21 11:26:42.074,0,True
3,2,Waste of money,"Very unhappy with this keyboard, it would slip...",[],B07899MFZ2,B07L5L22ZL,AGYEAZK4OEYF2MSSTGJ5WNJDVZKA,2018-09-29 22:39:47.708,1,True
4,5,Nice,Work great,[],B09JSMNZRG,B09LTX3SQX,AH67BI7JTOFR35HMZYFVOEHM4CPQ,2023-01-11 21:15:08.805,0,True


In [11]:
len(df_reviews)

122840

### Preprocess tile and features

In [12]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [26]:
# def count_tokens(text):
    
#     encoding = tiktoken.encoding_for_model("text-embedding-3-small")
    
#     return len(encoding.encode(text))

def count_tokens(row):

    encoding = tiktoken.encoding_for_model("text-embedding-3-small")

    return len(encoding.encode(row["preprocessed_data"]))


In [25]:
count_tokens("How are you doing today?")

6

In [27]:
df_reviews["preprocessed_data"] = df_reviews.apply(preprocess_reviews_data, axis = 1)
df_reviews["token_count"] = df_reviews.apply(count_tokens, axis = 1)

In [28]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,token_count
0,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True,Nixplay 10.1 touch screen digital picture fram...,255
1,5,Great so far...,"Speedy delivery, great sound and a great warra...",[],B08H1WNYTR,B0C72D4J46,AFANVB6MPHJTCTFOVIEBKLWZ2GVA,2022-07-05 14:40:48.001,0,True,"Great so far... Speedy delivery, great sound a...",27
2,5,Set up is not that easy.,Nice looking set but installation instructions...,[],B09XGX1GMJ,B09XGQQ98K,AHACLF2COQQE2V33ZFXQ7THZOJ2Q,2022-09-21 11:26:42.074,0,True,Set up is not that easy. Nice looking set but ...,18
3,2,Waste of money,"Very unhappy with this keyboard, it would slip...",[],B07899MFZ2,B07L5L22ZL,AGYEAZK4OEYF2MSSTGJ5WNJDVZKA,2018-09-29 22:39:47.708,1,True,Waste of money Very unhappy with this keyboard...,66
4,5,Nice,Work great,[],B09JSMNZRG,B09LTX3SQX,AH67BI7JTOFR35HMZYFVOEHM4CPQ,2023-01-11 21:15:08.805,0,True,Nice Work great,3


In [29]:
len(df_reviews)

122840

In [ ]:
# Filter out token counts less 8192: when using open ai's free embedding model
df_reviews = df_reviews[df_reviews["token_count"] < 8192]

In [31]:
len(df_reviews)

122840

In [32]:
total_tokens = df_reviews["token_count"].sum()

In [33]:
total_tokens

np.int64(6506026)

### Embed the text and add additonal fields to the payload of each vector for reviews

In [35]:
df_data_to_embed = df_reviews[["preprocessed_data", "parent_asin"]]

In [36]:
df_data_to_embed.head()

,preprocessed_data,parent_asin
0,Nixplay 10.1 touch screen digital picture fram...,B0BNXXNBB4
1,"Great so far... Speedy delivery, great sound a...",B0C72D4J46
2,Set up is not that easy. Nice looking set but ...,B09XGQQ98K
3,Waste of money Very unhappy with this keyboard...,B07L5L22ZL
4,Nice Work great,B09LTX3SQX


In [37]:
data_to_embed_reviews = df_data_to_embed.to_dict(orient="records")

In [39]:
data_to_embed_reviews

[{'preprocessed_data': "Nixplay 10.1 touch screen digital picture frame I purchased this digital frame on a treasure truck deal - it  works.  You will have to load the app and follow directions to bluetooth transfer pics you choose from your gallery to the frame.  I purchased an additional memory chip - but it turns out there is no place to install the chip so don't waste your time there.  You can set a regular time to 'run' the frame or manually turn it off and/or on.  You can 'shuffle' OR play all your uploaded pics.  You do not get to choose the 'fade' that happens between pics, but you do get to choose how long each pic stays on the frame before moving to the next pic. Picture quality is pretty good - about as good as the pics you download on to it.  It's close to an 8x10 frame.  Not sure why they call it a 'touch screen' cause I can't figure out what you can do by touching the screen - but ok. It comes with charging cord, regular plug in - there appears to be a mini USB port, but 

In [40]:
len(data_to_embed_reviews)

122840

In [42]:
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]

In [43]:
text_to_embed_reviews

["Nixplay 10.1 touch screen digital picture frame I purchased this digital frame on a treasure truck deal - it  works.  You will have to load the app and follow directions to bluetooth transfer pics you choose from your gallery to the frame.  I purchased an additional memory chip - but it turns out there is no place to install the chip so don't waste your time there.  You can set a regular time to 'run' the frame or manually turn it off and/or on.  You can 'shuffle' OR play all your uploaded pics.  You do not get to choose the 'fade' that happens between pics, but you do get to choose how long each pic stays on the frame before moving to the next pic. Picture quality is pretty good - about as good as the pics you download on to it.  It's close to an 8x10 frame.  Not sure why they call it a 'touch screen' cause I can't figure out what you can do by touching the screen - but ok. It comes with charging cord, regular plug in - there appears to be a mini USB port, but I haven't tried it.  H

In [44]:
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

Processed 500 of 122840
Processed 1000 of 122840
Processed 1500 of 122840
Processed 2000 of 122840
Processed 2500 of 122840
Processed 3000 of 122840
Processed 3500 of 122840
Processed 4000 of 122840
Processed 4500 of 122840
Processed 5000 of 122840
Processed 5500 of 122840
Processed 6000 of 122840
Processed 6500 of 122840
Processed 7000 of 122840
Processed 7500 of 122840
Processed 8000 of 122840
Processed 8500 of 122840
Processed 9000 of 122840
Processed 9500 of 122840
Processed 10000 of 122840
Processed 10500 of 122840
Processed 11000 of 122840
Processed 11500 of 122840
Processed 12000 of 122840
Processed 12500 of 122840
Processed 13000 of 122840
Processed 13500 of 122840
Processed 14000 of 122840
Processed 14500 of 122840
Processed 15000 of 122840
Processed 15500 of 122840
Processed 16000 of 122840
Processed 16500 of 122840
Processed 17000 of 122840
Processed 17500 of 122840
Processed 18000 of 122840
Processed 18500 of 122840
Processed 19000 of 122840
Processed 19500 of 122840
Proces

In [45]:
len(embeddings)

122840

In [46]:
pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding
            },
            payload=data
        )
    )
    i += 1

In [47]:
pointstructs[0].vector

{'text-embedding-3-small': [0.0068359375,
  0.01453399658203125,
  -0.055877685546875,
  -0.019622802734375,
  -0.01161956787109375,
  -0.03424072265625,
  0.0307769775390625,
  0.032440185546875,
  -0.0233001708984375,
  -0.025634765625,
  0.01971435546875,
  -0.01543426513671875,
  -0.042144775390625,
  0.01568603515625,
  0.0230560302734375,
  -0.034454345703125,
  -0.027008056640625,
  -0.038848876953125,
  -0.0162811279296875,
  0.0150604248046875,
  0.03546142578125,
  0.0186614990234375,
  0.018798828125,
  -0.052001953125,
  0.027679443359375,
  -0.004909515380859375,
  -0.01291656494140625,
  -0.0202789306640625,
  0.0265045166015625,
  0.03204345703125,
  -0.087890625,
  -0.0201873779296875,
  0.0003254413604736328,
  -0.05615234375,
  -0.01175689697265625,
  -0.048583984375,
  0.012969970703125,
  -0.031402587890625,
  -0.045684814453125,
  -0.023345947265625,
  0.0265045166015625,
  0.029998779296875,
  0.017852783203125,
  -0.01409149169921875,
  0.0275726318359375,
  -0.0

In [48]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

Processed 100 of 122840
Processed 200 of 122840
Processed 300 of 122840
Processed 400 of 122840
Processed 500 of 122840
Processed 600 of 122840
Processed 700 of 122840
Processed 800 of 122840
Processed 900 of 122840
Processed 1000 of 122840
Processed 1100 of 122840
Processed 1200 of 122840
Processed 1300 of 122840
Processed 1400 of 122840
Processed 1500 of 122840
Processed 1600 of 122840
Processed 1700 of 122840
Processed 1800 of 122840
Processed 1900 of 122840
Processed 2000 of 122840
Processed 2100 of 122840
Processed 2200 of 122840
Processed 2300 of 122840
Processed 2400 of 122840
Processed 2500 of 122840
Processed 2600 of 122840
Processed 2700 of 122840
Processed 2800 of 122840
Processed 2900 of 122840
Processed 3000 of 122840
Processed 3100 of 122840
Processed 3200 of 122840
Processed 3300 of 122840
Processed 3400 of 122840
Processed 3500 of 122840
Processed 3600 of 122840
Processed 3700 of 122840
Processed 3800 of 122840
Processed 3900 of 122840
Processed 4000 of 122840
Processed

### A function to run search against reviews on a prefiltered set of product IDs

In [49]:

def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )
    
    return results

In [50]:
reviews = retrieve_prefiltered_reviews_data("bad_quality", ["B09Q5TNDHY"])

In [51]:
reviews

QueryResponse(points=[ScoredPoint(id=62005, version=623, score=0.5, payload={'preprocessed_data': 'Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend', 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=80491, version=807, score=0.33333334, payload={'preprocessed_data': "Its stopped working It's not that durable", 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=43190, version=434, score=0.25, payload={'preprocessed_data': "Doesn't work step tracker to sensitive It's nice, decent features good screen... but step accuracy is way off . Did the dishes and said I walked 100 steps. Also walked 100 steps in my sleep", 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=105619, version=1059, score=0.2, payl

In [52]:
reviews.points

[ScoredPoint(id=62005, version=623, score=0.5, payload={'preprocessed_data': 'Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend', 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=80491, version=807, score=0.33333334, payload={'preprocessed_data': "Its stopped working It's not that durable", 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=43190, version=434, score=0.25, payload={'preprocessed_data': "Doesn't work step tracker to sensitive It's nice, decent features good screen... but step accuracy is way off . Did the dishes and said I walked 100 steps. Also walked 100 steps in my sleep", 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=105619, version=1059, score=0.2, payload={'preprocessed

In [53]:
reviews = retrieve_prefiltered_reviews_data("bad_quality", ["B09Q5TNDHY", "B0B4NJ8NKN" ])

In [54]:
reviews.points

[ScoredPoint(id=51290, version=515, score=0.5, payload={'preprocessed_data': 'Buena Buena', 'parent_asin': 'B0B4NJ8NKN'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=71239, version=715, score=0.33333334, payload={'preprocessed_data': 'Low quality piece of trash Adapters broke within 2 weeks of usage. I wouldn’t recommend these.', 'parent_asin': 'B0B4NJ8NKN'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=62005, version=623, score=0.25, payload={'preprocessed_data': 'Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend', 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=99289, version=995, score=0.2, payload={'preprocessed_data': 'Super cheap build and buzz when in use Super cheap build quality and buzz sound while using', 'parent_asin': 'B0B4NJ8N

### Define the reviews retrieval tool

In [55]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }
    
def process_reviews_context(context):

    formatted_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, user review: {chunk}\n"

    return formatted_context

def get_formatted_reviews_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:

    """Get the top k reviews matching a query for a list of prefiltered items.
    
    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered
    
    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    retrieved_context = retrieve_prefiltered_reviews_data(
        query,
        parent_asins,
        top_k
    )
    formatted_context = process_reviews_context(retrieved_context)

    return formatted_context

In [56]:
result = get_formatted_reviews_context("bad quality", ["B09Q5TNDHY", "B0B4NJ8NKN"])

In [57]:
print(result)

- ID: B0B4NJ8NKN, user review: Low quality piece of trash Adapters broke within 2 weeks of usage. I wouldn’t recommend these.
- ID: B0B4NJ8NKN, user review: Buena Buena
- ID: B09Q5TNDHY, user review: Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend
- ID: B0B4NJ8NKN, user review: Defective product. The charger side does not charge. Unfortunately, I did not use them within the return window time period. Don’t waste your money
- ID: B0B4NJ8NKN, user review: Quality product Very nice quality for price, would buy again

